# Élasticité-prix — Olist marketplace

Objectif : estimer l'élasticité-prix de la demande par catégorie de produits, comme première brique du moteur de tarification.

**Avant de lancer ce notebook**, téléchargez les fichiers Olist (Kaggle : "Brazilian E-Commerce Public Dataset by Olist") et mettez à jour les chemins dans la cellule 2 :
- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_products_dataset.csv`

Plan :
1. Chargement et fusion des tables
2. Construction du panel catégorie × période
3. Modèle log-log (OLS) — élasticité globale
4. Élasticité par catégorie (interaction + effets fixes)
5. Alternative ML (gradient boosting + dérivée numérique)
6. Du coefficient au prix recommandé (règle de Lerner)
7. Limites à garder en tête

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 1. Chargement et fusion des tables

In [2]:
# --- Mettez à jour ces chemins vers vos fichiers Olist ---
PATH_ORDERS = "../data/raw/olist_orders_dataset.csv"
PATH_ITEMS = "../data/raw/olist_order_items_dataset.csv"
PATH_PRODUCTS = "../data/raw/olist_products_dataset.csv"

orders = pd.read_csv(PATH_ORDERS, parse_dates=["order_purchase_timestamp"])
items = pd.read_csv(PATH_ITEMS)
products = pd.read_csv(PATH_PRODUCTS)

print(orders.shape, items.shape, products.shape)

(99441, 8) (112650, 7) (32951, 9)


In [3]:
# On garde uniquement les commandes livrées, pour ne pas polluer la demande avec des annulations
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

df = (
    items
    .merge(orders_delivered[["order_id", "order_purchase_timestamp"]], on="order_id", how="inner")
    .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
)

df = df.dropna(subset=["product_category_name", "price"])
df["price"] = df["price"].astype(float)
df["period"] = df["order_purchase_timestamp"].dt.to_period("W").dt.start_time  # semaine ; passez à "M" pour un panel mensuel

print(df.shape)
df.head()

(108660, 10)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_purchase_timestamp,product_category_name,period
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.900,13.290,2017-09-13 08:59:02,cool_stuff,2017-09-11
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.900,19.930,2017-04-26 10:53:06,pet_shop,2017-04-24
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.000,17.870,2018-01-14 14:33:31,moveis_decoracao,2018-01-08
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.990,12.790,2018-08-08 10:00:35,perfumaria,2018-08-06
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.900,18.140,2017-02-04 13:57:51,ferramentas_jardim,2017-01-30


## 2. Construction du panel catégorie × période

In [4]:
panel = (
    df.groupby(["product_category_name", "period"])
      .agg(
          quantity=("order_item_id", "count"),
          price=("price", "mean"),
          freight_value=("freight_value", "mean"),
      )
      .reset_index()
)

# On retire les cellules trop clairsemées (peu de ventes -> log(0) et bruit important)
panel = panel[panel["quantity"] >= 3].copy()

panel["log_quantity"] = np.log(panel["quantity"])
panel["log_price"] = np.log(panel["price"])
panel["month"] = panel["period"].dt.month

print(panel.shape)
panel.head()

(3179, 8)


,product_category_name,period,quantity,price,freight_value,log_quantity,log_price,month
3,agro_industria_e_comercio,2017-02-13,3,21.990,12.960,1.099,3.091,2
13,agro_industria_e_comercio,2017-08-28,4,22.000,34.150,1.386,3.091,8
22,agro_industria_e_comercio,2017-11-13,4,402.500,27.188,1.386,5.998,11
23,agro_industria_e_comercio,2017-11-20,11,959.055,39.347,2.398,6.866,11
24,agro_industria_e_comercio,2017-11-27,3,287.633,21.597,1.099,5.662,11


## 3. Modèle log-log — élasticité globale

In [5]:
model_simple = smf.ols("log_quantity ~ log_price", data=panel).fit()
print(model_simple.summary())
print(f"\nÉlasticité-prix globale (pooled) : {model_simple.params['log_price']:.2f}")

                            OLS Regression Results                            
Dep. Variable:           log_quantity   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     2.877
Date:                Tue, 11 Aug 2026   Prob (F-statistic):             0.0899
Time:                        11:39:25   Log-Likelihood:                -5103.3
No. Observations:                3179   AIC:                         1.021e+04
Df Residuals:                    3177   BIC:                         1.022e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.0680      0.154     19.981      0.0

In [6]:
model_fe = smf.ols(
    "log_quantity ~ log_price + freight_value + C(product_category_name) + C(month)",
    data=panel,
).fit()

print(f"Élasticité-prix (contrôlée, effets fixes catégorie/mois) : {model_fe.params['log_price']:.2f}")

Élasticité-prix (contrôlée, effets fixes catégorie/mois) : -0.02


## 4. Élasticité par catégorie (interaction)

In [7]:
model_interact = smf.ols(
    "log_quantity ~ log_price * C(product_category_name) + C(month)",
    data=panel,
).fit()

base_elasticity = model_interact.params["log_price"]
categories = panel["product_category_name"].unique()

elasticity_by_category = {}
for cat in categories:
    key = f"log_price:C(product_category_name)[T.{cat}]"
    interaction = model_interact.params.get(key, 0.0)
    elasticity_by_category[cat] = base_elasticity + interaction

elasticity_df = (
    pd.Series(elasticity_by_category, name="elasticite")
    .sort_values()
    .to_frame()
)
elasticity_df

,elasticite
portateis_cozinha_e_preparadores_de_alimentos,-2.663
relogios_presentes,-1.313
moveis_escritorio,-1.282
ferramentas_jardim,-0.931
papelaria,-0.807
...,...
beleza_saude,0.576
brinquedos,0.660
bebes,0.745
esporte_lazer,0.969


## 5. Alternative : élasticité locale via gradient boosting
Capture une élasticité potentiellement non linéaire selon le niveau de prix.

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

features = ["log_price", "freight_value", "month"]
cat_dummies = pd.get_dummies(panel["product_category_name"], prefix="cat", drop_first=True)
X = pd.concat([panel[features], cat_dummies], axis=1)
y = panel["log_quantity"]

gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X, y)

# Élasticité locale : petite variation de log_price, effet sur la prédiction
delta = 0.01
X_up = X.copy()
X_up["log_price"] += delta
panel["elasticity_ml"] = (gbr.predict(X_up) - gbr.predict(X)) / delta

panel.groupby("product_category_name")["elasticity_ml"].mean().sort_values()

product_category_name
beleza_saude                         -1.210
relogios_presentes                   -1.070
construcao_ferramentas_construcao    -0.899
agro_industria_e_comercio            -0.833
construcao_ferramentas_seguranca     -0.592
                                      ...  
artes                                 0.908
bebidas                               1.118
construcao_ferramentas_ferramentas    2.403
tablets_impressao_imagem              3.895
flores                                4.430
Name: elasticity_ml, Length: 68, dtype: float64

## 6. Du coefficient au prix recommandé (règle de Lerner)

(P* - Cm) / P* = -1/E, soit **P\* = Cm / (1 + 1/E)**.

Olist ne fournit aucune donnée de coût réelle — `cm_assumed` ci-dessous est une approximation à documenter clairement dans votre rapport.

In [9]:
def optimal_price(marginal_cost, elasticity):
    """Prix optimal (monopole) selon l'indice de Lerner.
    elasticity doit être négative (ex : -2.0) et < -1 (demande élastique)."""
    if elasticity >= -1:
        raise ValueError("La formule suppose une demande élastique (E < -1)")
    return marginal_cost / (1 + 1 / elasticity)

# Exemple sur la première catégorie du classement
category_example = elasticity_df.index[0]
e = elasticity_df.loc[category_example, "elasticite"]
avg_price = panel.loc[panel["product_category_name"] == category_example, "price"].mean()
cm_assumed = 0.6 * avg_price  # hypothèse à ajuster : coût marginal = 60% du prix moyen observé

print(f"Catégorie : {category_example}")
print(f"Élasticité estimée : {e:.2f}")
print(f"Prix moyen observé : {avg_price:.2f} BRL")
print(f"Prix recommandé (Cm approché à {cm_assumed:.2f} BRL) : {optimal_price(cm_assumed, e):.2f} BRL")

Catégorie : portateis_cozinha_e_preparadores_de_alimentos
Élasticité estimée : -2.66
Prix moyen observé : 22.03 BRL
Prix recommandé (Cm approché à 13.22 BRL) : 21.17 BRL


## 7. Limites à garder en tête

- **Endogénéité** : un vendeur peut augmenter son prix parce que la demande est forte (causalité inverse). Les effets fixes catégorie/mois réduisent ce biais mais ne l'éliminent pas — une correction complète demanderait une variable instrumentale.
- **Coût marginal** : Olist ne fournit aucune donnée de coût réelle. `cm_assumed` est une approximation à documenter clairement.
- **Volume clairsemé** : beaucoup de catégories/périodes ont trop peu de ventes pour une élasticité fiable — le filtre `quantity >= 3` est arbitraire, à ajuster et justifier.
- **Élasticité réalisée, pas élasticité de conversion** : on observe des ventes conclues, pas les vues ni les abandons de panier — contrairement à un vrai moteur type Amazon.

### Prochaine étape
Une fois l'élasticité validée, la phase 2 consiste à construire le moteur réactif (règles ou bandit contextuel) qui applique la règle de Lerner selon des signaux comme la vélocité de vente récente ou la position prix dans la catégorie.

## 8. Estimation causale avec DoubleML

La régression log-log ci-dessus suppose que les effets fixes catégorie/mois suffisent à neutraliser le confondant (la demande qui pousse à la fois le prix et les ventes). DoubleML fait la même chose mais avec deux modèles ML flexibles à la place d'une hypothèse linéaire :
1. Un modèle prédit `log_quantity` à partir des confondants
2. Un modèle prédit `log_price` à partir des confondants
3. On retire à chacun ce que les confondants expliquaient (les résidus), puis on régresse les résidus l'un sur l'autre — ce qu'il reste, c'est l'effet du prix "nettoyé" de la confusion

Le coefficient obtenu s'interprète exactement comme l'élasticité de la régression log-log, mais avec une hypothèse plus faible sur la façon dont les confondants agissent.

### 8.1 Vérification sur données synthétiques

Avant de faire confiance à la méthode sur Olist, on la teste sur des données simulées où l'on connaît la vraie élasticité — ça permet de vérifier que DoubleML corrige bien le biais, avant de l'appliquer à un cas où on ne connaît pas la réponse.

In [11]:
import numpy as np
import doubleml as dml
from sklearn.ensemble import RandomForestRegressor

# Données simulées : un confondant "demande" pousse à la fois le prix et la quantité,
# avec une vraie élasticité connue de -1.5
np.random.seed(42)
n_sim = 600
month_sim = np.random.randint(1, 13, n_sim)
demande_forte = np.sin(month_sim / 12 * 2 * np.pi) + np.random.normal(0, 0.3, n_sim)
log_price_sim = 4 + 0.4 * demande_forte + np.random.normal(0, 0.2, n_sim)
true_elasticity = -1.5
log_quantity_sim = 3 + true_elasticity * log_price_sim + 0.6 * demande_forte + np.random.normal(0, 0.2, n_sim)

df_sim = pd.DataFrame({"log_quantity": log_quantity_sim, "log_price": log_price_sim, "month": month_sim})

data_sim = dml.DoubleMLData(df_sim, y_col="log_quantity", d_cols="log_price", x_cols=["month"])
model_sim = dml.DoubleMLPLR(
    data_sim,
    ml_l=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    ml_m=RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    n_folds=5,
)
model_sim.fit()

naive_sim = np.polyfit(log_price_sim, log_quantity_sim, 1)[0]

print(f"Vraie élasticité simulée      : {true_elasticity:.2f}")
print(f"Régression naïve (biaisée)    : {naive_sim:.2f}")
print(f"DoubleML (corrigée)           : {model_sim.coef[0]:.2f}")

c:\Users\DELL\Desktop\Projet Data\Data Science\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vraie élasticité simulée      : -1.50
Régression naïve (biaisée)    : -0.45
DoubleML (corrigée)           : -1.04


### 8.2 Application au panel Olist

In [12]:
confounders = ["freight_value", "month"]
cat_dummies_dml = pd.get_dummies(panel["product_category_name"], prefix="cat", drop_first=True)
X_confounders = pd.concat([panel[confounders].reset_index(drop=True), cat_dummies_dml.reset_index(drop=True)], axis=1)

dml_df = pd.concat(
    [panel[["log_quantity", "log_price"]].reset_index(drop=True), X_confounders],
    axis=1,
)

data_dml = dml.DoubleMLData(
    dml_df,
    y_col="log_quantity",
    d_cols="log_price",
    x_cols=list(X_confounders.columns),
)

model_plr = dml.DoubleMLPLR(
    data_dml,
    ml_l=RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42),
    ml_m=RandomForestRegressor(n_estimators=300, max_depth=5, random_state=42),
    n_folds=5,
)
model_plr.fit()

print(model_plr.summary)

            coef  std err      t  P>|t|  2.5 %  97.5 %
log_price -0.029    0.038 -0.770  0.441 -0.103   0.045


In [13]:
comparison = pd.DataFrame({
    "méthode": ["OLS log-log (effets fixes)", "DoubleML (causal ML)"],
    "élasticité estimée": [model_fe.params["log_price"], model_plr.coef[0]],
})
comparison

,méthode,élasticité estimée
0,OLS log-log (effets fixes),-0.025
1,DoubleML (causal ML),-0.029


**Comment lire la comparaison :**
- Résultats proches → les effets fixes catégorie/mois captaient déjà l'essentiel de la confusion, la régression log-log était une approximation raisonnable
- Écart important → suggère une confusion non linéaire que les dummies de catégorie ne capturaient pas, et que RandomForest, lui, détecte

**Limites propres à DoubleML, à documenter :**
- Suppose toujours l'absence de confondant *non observé* — si un facteur qui influence à la fois prix et ventes n'est pas dans vos colonnes (`x_cols`), aucune méthode ne peut le corriger, causale ou non
- Le choix des modèles ML (`ml_l`, `ml_m`) influence le résultat — testez plusieurs configurations (profondeur d'arbre, gradient boosting à la place de RandomForest) pour vérifier la stabilité
- Demande plus de données que l'OLS pour rester stable (le cross-fitting divise l'échantillon en `n_folds` parties)